# Ranker Validation

Inspect scores, rationales, dealbreaker hits, and spot-check the threshold boundary.

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path

import json
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import select, func
from sqlalchemy.orm import Session
from src.db import get_engine, Posting

engine = get_engine(Path('../data/jobs.db'))
pd.set_option('display.max_colwidth', 100)

## Score distribution

In [ ]:
with Session(engine) as s:
    all_scored = s.scalars(
        select(Posting).where(Posting.relevance_score.is_not(None))
    ).all()

df = pd.DataFrame([
    {
        'id': p.id,
        'company': p.company,
        'title': p.title,
        'score': p.relevance_score,
        'level': p.level_match,
        'status': p.status,
        'rationale': p.rationale,
        'dealbreakers': json.loads(p.dealbreakers_hit or '[]'),
    }
    for p in all_scored
])

print(f'Total scored: {len(df)} | Ranked: {(df.status != "skipped").sum()} | Skipped: {(df.status == "skipped").sum()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score histogram
colours = ['#e74c3c' if s < 6 else '#4c9be8' for s in range(11)]
counts = df['score'].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, color=[colours[i] for i in counts.index])
axes[0].axvline(6, color='black', linestyle='--', linewidth=1, label='threshold (6)')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Postings')
axes[0].set_title('Score distribution (red = skipped)')
axes[0].legend()

# Level match breakdown
level_counts = df['level'].value_counts()
axes[1].pie(level_counts.values, labels=level_counts.index, autopct='%1.0f%%',
            colors=['#e74c3c', '#4c9be8', '#f0ad4e'])
axes[1].set_title('Level match')

plt.tight_layout()
plt.show()

## Dealbreaker analysis

In [ ]:
df_hits = df[df['dealbreakers'].apply(len) > 0].copy()
print(f'Postings with dealbreaker hits: {len(df_hits)}')

# Which dealbreakers fire most
from collections import Counter
all_hits = [h for hits in df_hits['dealbreakers'] for h in hits]
print('\nDealbreaker frequency:')
for db_id, count in Counter(all_hits).most_common():
    print(f'  {db_id}: {count}')

df_hits[['company', 'title', 'score', 'dealbreakers', 'rationale']].head(10)

## Threshold boundary — spot-check scores 4–7

These are the postings closest to the pass/fail line. Worth reviewing manually.

In [ ]:
boundary = df[df['score'].between(4, 7)].sort_values('score', ascending=False)
boundary[['score', 'status', 'level', 'company', 'title', 'rationale']].head(20)

## Top-ranked postings

In [ ]:
top = df[df['status'] != 'skipped'].sort_values('score', ascending=False).head(20)
top[['score', 'level', 'company', 'title', 'rationale']]

## Spot-check a specific posting

Change `posting_id` to inspect any row in detail.

In [ ]:
posting_id = df.sort_values('score', ascending=False).iloc[0]['id']  # default: highest-scored

with Session(engine) as s:
    p = s.get(Posting, int(posting_id))

print(f'ID:          {p.id}')
print(f'Company:     {p.company}')
print(f'Title:       {p.title}')
print(f'Location:    {p.location}')
print(f'Score:       {p.relevance_score}')
print(f'Level:       {p.level_match}')
print(f'Status:      {p.status}')
print(f'Dealbreakers: {json.loads(p.dealbreakers_hit or "[]")}')
print(f'Rationale:   {p.rationale}')
print(f'URL:         {p.url}')
print(f'\n--- Description (first 1000 chars) ---')
print(p.description[:1000])